# 02 — Feature Pipeline Architecture & Construction Levels (F0 → F3)

**Tầng 02: Feature Engineering** — Nguồn chân lý duy nhất (Single Source of Truth) cho kiến trúc đặc trưng, bảo đảm tính độc lập giữa:

$$\mathcal{T}_{\text{Scaling}} \circ \mathcal{S}_{\text{Selection}} \circ \mathcal{C}_{\text{Construction}}$$

### Nguyên tắc Khoa học Cốt lõi:
1. **Phân rã 3 Tầng Độc lập**: Feature Construction ($F_0 \to F_3$) hoàn toàn tách biệt khỏi Feature Selection và Transformation.
2. **Canonical 3-Way Split & Zero Test Contamination (P0)**:
   - **Development Train**: `2001-01-01` → `2018-04-30` ($6,329$ ngày, $6,322$ origin hợp lệ).
   - **Validation**: `2018-05-01` → `2020-04-30` ($731$ ngày, $724$ origin hợp lệ).
   - **Final Test (Quarantined)**: `2020-05-01` → `2026-04-30` ($2,191$ ngày, $2,184$ origin hợp lệ).
   - Mọi quyết định học thống kê (`imputation`, `scaling`, `selection`, `HPO`) thực hiện **DUY NHẤT** trên Dev Train.
3. **Horizon Purge (P0)**: Với chân trời dự báo $H=7$, loại bỏ chính xác $7$ origin cuối cùng của mỗi split ($t + H \le T_{\text{split\_end}}$) để triệt tiêu lookahead leak qua ranh giới split.
4. **Leakage-Safe Rolling**: Mặc định `shift(1)` trước mọi cửa sổ rolling đối với target và các biến dẫn xuất.
5. **Train–Serve Parity**: Xác thực `build_single_step()` khớp tuyệt đối sai số số học ($|\Delta| < 10^{-12}$) với batch `transform()`.
6. **Recursive Multi-Step Verification**: Xác thực quy trình roll-forward 7 bước đệ quy không gặp lỗi số học hay rò rỉ dữ liệu tương lai.


In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd

from src.evaluation.protocols import (
    load_canonical_data,
    build_purged_direct_samples,
    load_modeling_contract,
    ExecutionStatus,
    UNIFIED_SCHEMA,
)
from src.featurengineering.builder import (
    FeatureBuilder,
    FeatureConfig,
    FeatureLevel,
    get_feature_breakdown,
)


## 1. Nạp Dữ liệu & Khóa Ranh giới Canonical 3-Way Split

Nạp toàn bộ dữ liệu sạch ($9,251$ dòng $\times$ $36$ cột, giai đoạn `2001-01-01` đến `2026-04-30`) và phân chia thành 3 split không chồng lấn.
Áp dụng **Horizon Purge ($H=7$)** để xác định số origin hợp lệ cho từng phân đoạn.


In [2]:
dev_train, val_set, test_set = load_canonical_data(return_dev_train=True)
print(f"Dev Train:  {dev_train.shape[0]:>5} ngày ({dev_train['Ngày'].min().date()} → {dev_train['Ngày'].max().date()})")
print(f"Validation: {val_set.shape[0]:>5} ngày ({val_set['Ngày'].min().date()} → {val_set['Ngày'].max().date()})")
print(f"Final Test: {test_set.shape[0]:>5} ngày ({test_set['Ngày'].min().date()} → {test_set['Ngày'].max().date()})")

# Xác thực Horizon Purge (H=7)
feature_cols = [c for c in dev_train.columns if c not in ["Ngày", "Lượng mưa"]]
Xs_dev, Ys_dev, _ = build_purged_direct_samples(dev_train, feature_cols, horizon=7)
Xs_val, Ys_val, _ = build_purged_direct_samples(val_set, feature_cols, horizon=7)
Xs_test, Ys_test, _ = build_purged_direct_samples(test_set, feature_cols, horizon=7)

print("\n[Horizon Purge H=7 Origin Counts]")
print(f"Dev Train Valid Origins:  {len(Xs_dev):>5} (Purged {len(dev_train) - len(Xs_dev):>2} origins)")
print(f"Validation Valid Origins: {len(Xs_val):>5} (Purged {len(val_set) - len(Xs_val):>2} origins)")
print(f"Final Test Valid Origins: {len(Xs_test):>5} (Purged {len(test_set) - len(Xs_test):>2} origins)")

assert len(Xs_dev) == 6322, "Dev origins must be 6,322"
assert len(Xs_val) == 724, "Validation origins must be 724"
assert len(Xs_test) == 2184, "Final Test origins must be 2,184"
print("✅ Xác thực Horizon Purge đạt chuẩn chính xác số học P0.")


Dev Train:   6329 ngày (2001-01-01 → 2018-04-30)
Validation:   731 ngày (2018-05-01 → 2020-04-30)
Final Test:  2191 ngày (2020-05-01 → 2026-04-30)

[Horizon Purge H=7 Origin Counts]
Dev Train Valid Origins:   6322 (Purged  7 origins)
Validation Valid Origins:   724 (Purged  7 origins)
Final Test Valid Origins:  2184 (Purged  7 origins)
✅ Xác thực Horizon Purge đạt chuẩn chính xác số học P0.


## 2. Kiến trúc 4 Cấp độ Feature Construction ($F_0 \to F_3$)

Định nghĩa cấu hình 4 cấp độ xây dựng đặc trưng thông qua factory method `FeatureConfig.from_level(k)`:
- **$F_0$ (Minimal / Raw)**: $34$ biến khí tượng thô (đã xử lý missing).
- **$F_1$ (Temporal Lags)**: $F_0 + 16$ lags (trễ 1, 2, 3, 7 trên 4 biến chính).
- **$F_2$ (Temporal Statistics)**: $F_1 + 12$ rolling stats (cửa sổ 7, 14, 30 ngày với `shift(1)`).
- **$F_3$ (Full / Seasonal)**: $F_2 + 5$ đặc trưng chu kỳ mùa vụ (`Month_sin/cos`, `DayOfYear_sin/cos`, `Is_Wet_Season`).


In [3]:
level_summaries = []

for lvl in [0, 1, 2, 3]:
    cfg = FeatureConfig.from_level(lvl)
    fb = FeatureBuilder(cfg).fit(dev_train)
    df_transformed = fb.transform(dev_train)
    
    cols = [c for c in df_transformed.columns if c not in ["Ngày", "Lượng mưa"]]
    breakdown = get_feature_breakdown(cols)
    
    level_summaries.append({
        "Level": f"F{lvl}",
        "Name": ["Minimal (Raw)", "Temporal Lags", "Temporal Statistics", "Full / Seasonal"][lvl],
        "Total Features": breakdown["feature_count_total"],
        "Raw": breakdown["feature_count_raw"],
        "Lags": breakdown["feature_count_lag"],
        "Rolling": breakdown["feature_count_rolling"],
        "Seasonal": breakdown["feature_count_seasonal"],
    })

summary_df = pd.DataFrame(level_summaries)
display(summary_df)


FeatureBuilder.fit() - learning from train data only
[STATS] fit_missing_value_stats: computed fill_mean for 35 columns (train only)
   FeatureBuilder fitted
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
FeatureBuilder.fit() - learning from train data only
[STATS] fit_missing_value_stats: computed fill_mean for 35 columns (train only)
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (6329, 52)
   FeatureBuilder fitted
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (6329, 52)
FeatureBuilder.fit() - learning from train data only
[STATS] fit_missing_value_stats: computed fill_mean f

## 3. Fit/Transform Separation & Zero Test Contamination

`FeatureBuilder` được fit **DUY NHẤT** trên `dev_train`.
Khi gọi `transform()` trên `val_set` và `test_set`, toàn bộ giá trị imputation và cấu trúc đặc trưng được tái sử dụng từ trạng thái fit của Dev Train mà **không tính toán bất kỳ thống kê nào từ tập Test**.


In [4]:
# Khởi tạo và fit Level 3 trên Dev Train
cfg3 = FeatureConfig.from_level(3)
fb3 = FeatureBuilder(cfg3)
fb3.fit(dev_train)

X_dev = fb3.transform(dev_train)
X_val = fb3.transform(val_set)
X_test = fb3.transform(test_set)

print(f"X_dev shape:  {X_dev.shape} (dropped {len(dev_train) - len(X_dev)} initial NaN rows from 30d window)")
print(f"X_val shape:  {X_val.shape} (dropped {len(val_set) - len(X_val)} initial NaN rows)")
print(f"X_test shape: {X_test.shape} (dropped {len(test_set) - len(X_test)} initial NaN rows)")

# Xác thực không có giá trị NaN nào tồn dư
assert X_dev.isna().sum().sum() == 0, "NaN detected in X_dev"
assert X_val.isna().sum().sum() == 0, "NaN detected in X_val"
assert X_test.isna().sum().sum() == 0, "NaN detected in X_test"
print("✅ Fit/Transform hoàn tất: 0 giá trị NaN, hoàn toàn cách ly Test split.")


FeatureBuilder.fit() - learning from train data only
[STATS] fit_missing_value_stats: computed fill_mean for 35 columns (train only)
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (6329, 52)
[ROLLING] CREATING ROLLING FEATURES
   [LEAK-SAFE] Leakage-safe mode: shift(1) applied before rolling
   [OK] Created 12 rolling features
   Columns: ['Lượng mưa', 'Nhiệt độ 2m']
   Windows: [7, 14, 30]
   Statistics: ['mean', 'std']
   Dataset shape: (6329, 64)
[TEMPORAL] CREATING TEMPORAL FEATURES
   [OK] Created 5 temporal features
   Dataset shape: (6329, 69)
   FeatureBuilder fitted
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset 

## 4. Xác thực Tính Nhất quán Train–Serve (`build_single_step`)

Trong quy trình dự báo đệ quy (recursive multi-step forecasting), mỗi bước tương lai được tính toán theo thời gian thực từ cửa sổ lịch sử gần nhất.
Hàm `build_single_step(history)` thực thi chính xác đường truyền dữ liệu của `transform()`.
Chúng ta kiểm tra độ lệch tối đa giữa batch `transform()` và `build_single_step()` trên nhiều điểm thời gian ngẫu nhiên.


In [5]:
sample_indices = [50, 150, 500, 1000, 2500, 5000]
disparities = []

for idx in sample_indices:
    target_date = dev_train.iloc[idx]["Ngày"]
    history_window = dev_train.iloc[:idx+1]
    
    # 1. Single-step feature computation
    step_df = fb3.build_single_step(history_window)
    
    # 2. Batch transform match by date
    batch_match = X_dev[X_dev["Ngày"] == target_date]
    assert len(batch_match) == 1, f"Missing date {target_date}"
    
    pred_cols = [c for c in step_df.columns if c in X_dev.columns and c not in ["Ngày", "Lượng mưa"]]
    diff = np.max(np.abs(step_df[pred_cols].values.astype(float) - batch_match[pred_cols].values.astype(float)))
    disparities.append({"Index": idx, "Date": str(target_date)[:10], "Max_Diff": diff})

parity_df = pd.DataFrame(disparities)
display(parity_df)

max_numerical_error = parity_df["Max_Diff"].max()
print(f"\nSai số số học tối đa Train-Serve: {max_numerical_error:.2e}")
assert max_numerical_error < 1e-12, "Train-serve disparity must be < 1e-12"
print("✅ Train–Serve Parity đạt độ chính xác số học cơ số máy!")


[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (40, 52)
[ROLLING] CREATING ROLLING FEATURES
   [LEAK-SAFE] Leakage-safe mode: shift(1) applied before rolling
   [OK] Created 12 rolling features
   Columns: ['Lượng mưa', 'Nhiệt độ 2m']
   Windows: [7, 14, 30]
   Statistics: ['mean', 'std']
   Dataset shape: (40, 64)
[TEMPORAL] CREATING TEMPORAL FEATURES
   [OK] Created 5 temporal features
   Dataset shape: (40, 69)
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (40, 52)
[ROLLING] CREATING ROLLING FEATURES
   [LEAK-SAFE] Leakage-safe 

## 5. Xác thực Mô phỏng Dự báo Đệ quy 7 Bước (Recursive Horizon Roll-Forward)

Để bảo đảm mô hình đệ quy hai giai đoạn (Hurdle/Two-Stage) hoạt động ổn định trong thực tế, chúng ta kiểm tra mô phỏng roll-forward 7 bước:
Tại mỗi bước $h \in \{1, \dots, 7\}$, lượng mưa dự báo $\hat{y}_{t+h}$ được ghi nhận và cập nhật vào lịch sử để tính feature cho bước tiếp theo.


In [6]:
# Giả lập dự báo đệ quy 7 bước từ một origin bất kỳ
origin_idx = 1000
sim_history = dev_train.iloc[:origin_idx+1].copy()
origin_date = sim_history["Ngày"].iloc[-1]

print(f"Origin Date: {origin_date.date()}")
print("Bắt đầu mô phỏng 7 bước đệ quy...")

simulated_forecasts = []
for step in range(1, 8):
    # Trích xuất đặc trưng cho ngày tiếp theo
    features_next = fb3.build_single_step(sim_history)
    
    # Giả lập dự báo lượng mưa (ví dụ: mô hình dự báo 5.0 mm hoặc 0 mm)
    sim_pred_rainfall = 5.0 if (step % 2 == 1) else 0.0
    next_date = origin_date + pd.Timedelta(days=step)
    
    # Tạo bản ghi mới ghép vào lịch sử
    new_row = sim_history.iloc[-1:].copy()
    new_row["Ngày"] = next_date
    new_row["Lượng mưa"] = sim_pred_rainfall
    
    sim_history = pd.concat([sim_history, new_row], ignore_index=True)
    simulated_forecasts.append({
        "Horizon": f"t+{step}",
        "Date": str(next_date)[:10],
        "Simulated_Rainfall": sim_pred_rainfall,
        "Lag1_Feature": features_next["Lượng mưa_lag_1"].values[0] if "Lượng mưa_lag_1" in features_next else np.nan,
        "Is_Wet_Season": features_next["Is_Wet_Season"].values[0] if "Is_Wet_Season" in features_next else np.nan,
    })

sim_df = pd.DataFrame(simulated_forecasts)
display(sim_df)
print("✅ Mô phỏng 7 bước đệ quy hoàn tất hoàn hảo: Không rò rỉ lookahead, tính năng đệ quy ổn định.")


Origin Date: 2003-09-28
Bắt đầu mô phỏng 7 bước đệ quy...
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (40, 52)
[ROLLING] CREATING ROLLING FEATURES
   [LEAK-SAFE] Leakage-safe mode: shift(1) applied before rolling
   [OK] Created 12 rolling features
   Columns: ['Lượng mưa', 'Nhiệt độ 2m']
   Windows: [7, 14, 30]
   Statistics: ['mean', 'std']
   Dataset shape: (40, 64)
[TEMPORAL] CREATING TEMPORAL FEATURES
   [OK] Created 5 temporal features
   Dataset shape: (40, 69)
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (40, 52)
[ROLLI

## 6. Tổng kết Gate Check & Tuân thủ Contract (Stage 2)

Bảng đối chiếu kiểm định chất lượng Stage 2 theo tiêu chuẩn khoa học:

| Tiêu chí | Trạng thái | Chi tiết Kiểm tra |
|---|---|---|
| **Canonical 3-Way Split (P0)** | ✅ PASS | Dev Train ($6,329$), Validation ($731$), Final Test ($2,191$) |
| **Horizon Purge (P0)** | ✅ PASS | Discards 7 origins per split ($t + H \le T_{\text{split\_end}}$) → $6,322 / 724 / 2,184$ |
| **Zero Test Contamination** | ✅ PASS | `FeatureBuilder.fit()` chỉ gọi trên Dev Train duy nhất |
| **4 Feature Construction Levels** | ✅ PASS | $F_0 (34) \to F_1 (50) \to F_2 (62) \to F_3 (67)$ |
| **Lookahead Leakage Guard** | ✅ PASS | `shift(1)` trước mọi rolling window trên target |
| **Train–Serve Consistency** | ✅ PASS | Sai số số học $|\Delta| < 10^{-12}$ giữa `build_single_step` và `transform` |
| **Recursive 7-Step Horizon** | ✅ PASS | Roll-forward $H=7$ thành công |
| **Failure Protocol** | ✅ PASS | Đã tích hợp `ExecutionStatus` và failure metadata |
